In [1]:
import numpy as np
import helper

In [2]:
rng = np.random.default_rng(42)
y = rng.permutation(10)


In [3]:
# =============================
# Exponential smoothing
# =============================
def exp_smoothing_forecast(y, alpha):
    """One-step-ahead simple exponential smoothing forecast."""
    es = np.empty_like(y, dtype=float)   # Allocate forecast array
    es[0] = y[0]                          # Initialize level

    for t in range(1, len(y)):           # Loop over time
        es[t] = alpha * y[t - 1] + (1.0 - alpha) * es[t - 1]  # Update rule
    es[0] = np.nan                        # Forecast undefined at first index
    return es                             # Return forecast series


In [4]:
len(y)

10

In [5]:
exp_smoothing_forecast(y, 0.1)

array([       nan, 5.        , 5.1       , 4.59      , 4.831     ,
       4.6479    , 4.38311   , 4.344799  , 4.8103191 , 4.42928719])

In [6]:
def estimate_alpha_exponential_smoothing(y, criterion="MSE", grid_size=2000, eps=1e-4):
    y = np.asarray(y, dtype=float)
    T = len(y)

    alphas = np.linspace(eps, 1.0, grid_size)

    alpha_t = np.full(T, np.nan)  # store alpha used to forecast y[t]
    yhat_t  = np.full(T, np.nan)  # yhat_t[t] forecasts y[t]
    u_t     = np.full(T, np.nan)  # u_t[t] = y[t] - yhat_t[t]

    # At time t, estimate alpha from y[:t+1], then forecast y[t+1]
    for t in range(2, T - 1):   
        y_sub = y[:t+1]         # data available up to time t

        best_alpha = None
        best_val = np.inf

        for a in alphas:
            yhat_sub = exp_smoothing_forecast(y_sub, a)
            u_sub = helper.forecast_errors(y_sub, yhat_sub)
            m = helper.forecast_metrics(u_sub, y_sub, tau=2)

            if criterion == "ME":
                val = m["ME"]
            elif criterion == "MAE":
                val = m["MAE"]
            elif criterion == "MAPE":
                val = m["MAPE"]
            elif criterion == "MSE":
                val = m["MSE"]
            else:
                raise ValueError("criterion must be one of: 'ME', 'MAE', 'MAPE', 'MSE'")

            if val < best_val:
                best_val = val
                best_alpha = a

        # Now forecast y[t+1] using alpha estimated from y[:t+1]
        y_sub_t = y[:t+2]  # exclude y[t+2] 
        yhat_sub_best = exp_smoothing_forecast(y_sub_t, best_alpha)

        print(
            f"# ={t+1:3d} | "
            f"len(y_sub)={len(y_sub):3d} | "
            f"len(yhat_sub)={len(yhat_sub_best):3d} | "
        )

        yhat_t[t+1] = yhat_sub_best[-1]     # forecast for index t+1
        alpha_t[t+1] = best_alpha
        u_t[t+1] = y[t+1] - yhat_t[t+1]

    return alpha_t, yhat_t, u_t


In [7]:
alpha_est, yhat_best, u_best = estimate_alpha_exponential_smoothing(
    y, criterion="MSE"
)

# =  3 | len(y_sub)=  3 | len(yhat_sub)=  4 | 
# =  4 | len(y_sub)=  4 | len(yhat_sub)=  5 | 
# =  5 | len(y_sub)=  5 | len(yhat_sub)=  6 | 
# =  6 | len(y_sub)=  6 | len(yhat_sub)=  7 | 
# =  7 | len(y_sub)=  7 | len(yhat_sub)=  8 | 
# =  8 | len(y_sub)=  8 | len(yhat_sub)=  9 | 
# =  9 | len(y_sub)=  9 | len(yhat_sub)= 10 | 


c:\Bao\FM-week1-1.24\AssignmentI\helper.py:797: RuntimeWarning: divide by zero encountered in divide
  mape = (100.0 * np.abs(u) / np.abs(y)).mean()  # MAPE


In [8]:
yhat_best

array([       nan,        nan,        nan, 4.99959999, 4.99980003,
       4.99960005, 4.74905848, 4.47398685, 4.99960024, 4.99920028])

In [9]:
def estimate_alpha_beta_holt_winters(y, criterion="SSE", grid_n=201, eps=1e-3):
    y = np.asarray(y, dtype=float)
    T = len(y)

    grid = np.linspace(eps, 1.0 - eps, grid_n)

    alpha_t = np.full(T, np.nan)
    beta_t = np.full(T, np.nan)
    yhat_t = np.full(T, np.nan)
    u_t = np.full(T, np.nan)
    loss_t = np.full(T, np.nan)

    # Need enough data to initialize Holt inside holt_fitted_forecast_series (uses y[0] and y[1])
    # and to produce a one-step-ahead forecast for index t (so t must be at least 2).
    for t in range(2, T - 1):
        y_sub = y[:t+2]  

        best_alpha = None
        best_beta = None
        best_loss = np.inf

        for alpha in grid:
            for beta in grid:
                F = helper.holt_fitted_forecast_series(y_sub, alpha, beta)

                y_eval = y_sub[1:]
                F_eval = F[1:]
                u = y_eval - F_eval

                if criterion == "ME":
                    loss = float(np.mean(u))
                elif criterion == "MAE":
                    loss = float(np.mean(np.abs(u)))
                elif criterion == "MAPE":
                    loss = float(np.mean(100.0 * np.abs(u) / np.abs(y_eval)))
                elif criterion == "MSE":
                    loss = float(np.mean(u ** 2))
                elif criterion == "SSE":
                    loss = float(np.sum(u ** 2))
                else:
                    raise ValueError("criterion must be one of: 'ME', 'MAE', 'MAPE', 'MSE', 'SSE'")

                if loss < best_loss:
                    best_alpha = alpha
                    best_beta = beta
                    best_loss = loss

        # One-step-ahead forecast for index t using best (alpha, beta) fitted on y[:t]
        F_best = helper.holt_fitted_forecast_series(y_sub, best_alpha, best_beta)
        yhat_t[t+1] = F_best[-1]
        u_t[t+1] = y[t] - yhat_t[t]

        alpha_t[t+1] = best_alpha
        beta_t[t+1] = best_beta
        loss_t[t+1] = best_loss

    return alpha_t, beta_t, yhat_t, u_t, loss_t

In [10]:
alpha_path, beta_path, F_hat, u_hat, loss_path = estimate_alpha_beta_holt_winters(
    y,
    criterion="SSE",
    grid_n=201,
    eps=1e-3,
)

In [11]:
F_hat

array([       nan,        nan,        nan, 7.00006234, 5.4002076 ,
       3.87244702, 2.57809164, 5.25321246, 4.09701714, 4.11631012])